# 🏥 Clinical Decision Support (CDS) — Vector Embedding & ChromaDB Indexing

This notebook loads section-aware chunks from `data/processed_chunks.json`, computes dense biomedical embeddings using **`pritamdeka/S-PubMedBert-MS-MARCO`**, and indexes them into a persistent **ChromaDB** vector database with complete clinical metadata.

### **Key Highlights**:
- **Biomedical S-PubMedBERT Model**: Pre-trained on PubMed and fine-tuned for MS-MARCO clinical passage retrieval.
- **Section-Context Preservation**: Embeds `embedding_text` containing section headers for optimal semantic grounding.
- **ChromaDB Vector Store**: Persisted locally at `data/chroma_db` with cosine similarity indexing.
- **Metadata Schema**: Preserves Document Name, Section Number/Title, Page Number, Evidence Grade, and Target Population.

In [ ]:
# Step 1: Install dependencies (run this if executing on Google Colab)
# !pip install sentence-transformers chromadb pydantic torch

In [ ]:
import os
import json
from pathlib import Path
import torch
import chromadb
from sentence_transformers import SentenceTransformer

# Determine project root path automatically
CURRENT_DIR = Path(os.getcwd())
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed_chunks.json"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"
COLLECTION_NAME = "clinical_guidelines"
MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"

# Hardware Acceleration (CUDA GPU / CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Project Root: {PROJECT_ROOT}")
print(f"[+] Chunks File:  {CHUNKS_FILE}")
print(f"[+] Chroma DB:    {CHROMA_DIR}")
print(f"[+] Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Step 2: Load processed section-aware chunks
if not CHUNKS_FILE.exists():
    raise FileNotFoundError(f"Chunks file not found at {CHUNKS_FILE}. Please run src/ingestion.py first.")

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"[+] Successfully loaded {len(chunks)} chunks from {CHUNKS_FILE.name}")
print("=" * 60)
print(f"Sample Chunk ID:       {chunks[0]['chunk_id']}")
print(f"Sample Document:       {chunks[0]['document_name']}")
print(f"Sample Section:        {chunks[0]['section_number']} - {chunks[0]['section_title']}")
print(f"Sample Evidence Grade: {chunks[0]['evidence_grade']}")
print(f"Sample Population:     {chunks[0]['target_population']}")
print("=" * 60)

In [ ]:
# Step 3: Load Domain-Adapted Biomedical Model & Generate Embeddings
print(f"[+] Loading '{MODEL_NAME}' on {device}...")
model = SentenceTransformer(MODEL_NAME, device=device)

# Use embedding_text (includes section headers) for optimal semantic retrieval
texts_to_embed = [chunk.get("embedding_text", chunk["content"]) for chunk in chunks]

print(f"[+] Generating dense embeddings for {len(texts_to_embed)} chunks...")
embeddings = model.encode(
    texts_to_embed,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Cosine similarity via normalized inner product
)

print(f"\n[+] Embedding matrix shape: {embeddings.shape} (Chunks x Vector Dim)")

In [ ]:
# Step 4: Index & Store in Persistent ChromaDB
os.makedirs(CHROMA_DIR, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Re-create clean collection with cosine distance metric
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"[+] Reset existing collection '{COLLECTION_NAME}'")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# Format records for ChromaDB
ids = [chunk["chunk_id"] for chunk in chunks]
documents = [chunk["content"] for chunk in chunks]
metadatas = [
    {
        "document_name": str(chunk.get("document_name", "")),
        "section_number": str(chunk.get("section_number", "")),
        "section_title": str(chunk.get("section_title", "")),
        "page_number": int(chunk.get("page_number", 1)),
        "evidence_grade": str(chunk.get("evidence_grade", "N/A")),
        "target_population": str(chunk.get("target_population", "Adults")),
        "char_count": int(chunk.get("char_count", len(chunk["content"]))),
    }
    for chunk in chunks
]
embeddings_list = embeddings.tolist()

# Batch insertion
BATCH_SIZE = 64
for i in range(0, len(ids), BATCH_SIZE):
    end_i = min(i + BATCH_SIZE, len(ids))
    collection.add(
        ids=ids[i:end_i],
        embeddings=embeddings_list[i:end_i],
        documents=documents[i:end_i],
        metadatas=metadatas[i:end_i]
    )

print(f"[+] Indexing complete: {collection.count()} chunks stored in ChromaDB at '{CHROMA_DIR}'!")

## 🧪 Step 5: Verification & Semantic Retrieval Test

Let's test semantic similarity queries against our indexed guidelines to confirm retrieval precision and metadata surfacing.

In [ ]:
def query_guidelines(query_str: str, top_k: int = 3):
    print("\n" + "=" * 80)
    print(f"CLINICAL QUERY: \"{query_str}\"")
    print("=" * 80)
    
    query_vec = model.encode([query_str], normalize_embeddings=True).tolist()
    
    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    for rank in range(top_k):
        c_id = results["ids"][0][rank]
        content = results["documents"][0][rank]
        meta = results["metadatas"][0][rank]
        dist = results["distances"][0][rank]
        sim = 1.0 - dist  # Cosine similarity score
        
        print(f"\n[Result #{rank+1}] Similarity: {sim:.4f} | Chunk: {c_id}")
        print(f"  - Document:          {meta['document_name']} (Page {meta['page_number']})")
        print(f"  - Section:           {meta['section_number']} — {meta['section_title']}")
        print(f"  - Evidence Grade:    {meta['evidence_grade']}")
        print(f"  - Target Population: {meta['target_population']}")
        print(f"  - Excerpt:           {content[:250].strip()}...")

# Test 1: Hip Fracture Analgesia (NICE CG124)
query_guidelines("What is the recommended analgesia regimen for hip fracture patients upon admission?", top_k=2)

# Test 2: Osteoporosis Screening in Older Women (USPSTF)
query_guidelines("What are the screening recommendations for osteoporosis in women aged 65 and older?", top_k=2)

# Test 3: Surgical Timing (NICE CG124)
query_guidelines("What is the recommended surgical timing for medically fit hip fracture patients?", top_k=2)